
---
## Section 5 - Implementation & Training


In [ ]:

# 5.1 M0 - Majority Class Baseline
baseline = DummyClassifier(strategy='most_frequent', random_state=42)
baseline.fit(X_train_sc, y_train)
print(" M0 - Majority Class Baseline trained.")


 M0 - Majority Class Baseline trained.


In [ ]:

# 5.2 M1 - Logistic Regression
# L2 regularisation (C=1.0), lbfgs solver (handles multicollinearity well),
# max_iter=1000 ensures convergence on scaled features.
lr_model = LogisticRegression(
    penalty='l2',
    C=1.0,
    solver='lbfgs',
    max_iter=1000,
    random_state=42
)
lr_model.fit(X_train_sc, y_train)
print(" M1 - Logistic Regression trained.")
print(f"   Converged in {lr_model.n_iter_[0]} iterations.")
print("\nLearned weights (feature importances via |w|):")
coef_df = pd.DataFrame({'Feature': FEATURES, 'Weight': lr_model.coef_[0]})
coef_df['|Weight|'] = coef_df['Weight'].abs()
print(coef_df.sort_values('|Weight|', ascending=False).to_string(index=False))


 M1 - Logistic Regression trained.
   Converged in 10 iterations.

Learned weights (feature importances via |w|):
             Feature    Weight  |Weight|
    ta_hist_win_rate  0.259793  0.259793
  tb_hist_avg_rating -0.234386  0.234386
    hist_rating_diff  0.163892  0.163892
  hist_win_rate_diff  0.124039  0.124039
 tb_hist_map_win_pct -0.111181  0.111181
 ta_hist_map_win_pct -0.108101  0.108101
is_elimination_match  0.094437  0.094437
    tb_hist_win_rate  0.090249  0.090249
        ta_ban_first -0.085617  0.085617
      is_grand_final  0.041680  0.041680
  ta_hist_avg_rating -0.020910  0.020910
        stage_stakes -0.019234  0.019234
    map_win_pct_diff  0.007177  0.007177


In [ ]:

# 5.3 M2 - Random Forest
# max_depth=6: selected via sensitivity analysis (Section 7) - balances
# train/test F1 gap.
# min_samples_leaf=5: prevents splits on noise, particularly important for
# small 2023 training subset.
# class_weight='balanced': corrects for the mild class imbalance.
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
print(" M2 - Random Forest trained.")
print(f"   n_estimators={rf_model.n_estimators}, max_depth={rf_model.max_depth}")


 M2 - Random Forest trained.
   n_estimators=300, max_depth=6


In [ ]:

# 5.4 M3 - XGBoost
# learning_rate=0.001: low shrinkage - converges slowly but generalises better
# across the 2023→2025 distribution shift.
# n_estimators=1000: compensates for low learning rate.
# max_depth=4: shallow trees reduce overfitting while capturing interactions.
# subsample=0.8, colsample_bytree=0.8: stochastic boosting improves ensemble
# diversity and robustness.
xgb_model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.001,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)
xgb_model.fit(X_train, y_train)
print(" M3 - XGBoost trained.")
print(f"   n_estimators={xgb_model.n_estimators}, learning_rate={xgb_model.learning_rate}")


 M3 - XGBoost trained.
   n_estimators=1000, learning_rate=0.001
